In [ ]:
# Install the Python packages required by this notebook.
%pip install -q duckdb pyarrow scikit-learn xgboost scipy joblib

In [ ]:
# Mount Google Drive so this notebook can access the private MIMIC-IV data and derived files.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Define the MIMIC-IV folders and the shared derived-data folder used by all notebooks.
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/Early Acute Kidney Injury Prediction + Production Monitoring/data")
HOSP_DIR = DATA_ROOT / "hosp"
ICU_DIR = DATA_ROOT / "icu"
DERIVED_DIR = DATA_ROOT / "derived"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the libraries used to evaluate discrimination, calibration, thresholds, and subgroup performance.
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [ ]:
# Load the trained models, feature list, and validation and test cohorts.
MODEL_DIR = DERIVED_DIR / "models"

logistic_model = joblib.load(MODEL_DIR / "logistic_regression.joblib")
xgboost_model = joblib.load(MODEL_DIR / "xgboost.joblib")

with open(MODEL_DIR / "feature_columns.json") as file:
    feature_columns = json.load(file)

validation_df = pd.read_parquet(DERIVED_DIR / "validation.parquet")
test_df = pd.read_parquet(DERIVED_DIR / "test.parquet")

In [ ]:
# Define helper functions for threshold selection and clinically useful binary-classification metrics.
def choose_threshold_for_sensitivity(y_true, y_probability, target_sensitivity=0.85):
    thresholds = np.linspace(0.0, 1.0, 1001)
    valid_thresholds = []

    for threshold in thresholds:
        prediction = (y_probability >= threshold).astype(int)
        sensitivity = recall_score(
            y_true,
            prediction,
            zero_division=0,
        )

        if sensitivity >= target_sensitivity:
            valid_thresholds.append(threshold)

    return max(valid_thresholds) if valid_thresholds else 0.5

def classification_metrics(y_true, y_probability, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_probability = np.asarray(y_probability)
    y_prediction = (y_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_prediction,
        labels=[0, 1],
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan

    return {
        "n": int(len(y_true)),
        "prevalence": float(y_true.mean()),
        "threshold": float(threshold),
        "auroc": float(roc_auc_score(y_true, y_probability)),
        "auprc": float(average_precision_score(y_true, y_probability)),
        "sensitivity": float(recall_score(y_true, y_prediction, zero_division=0)),
        "specificity": float(specificity),
        "ppv": float(precision_score(y_true, y_prediction, zero_division=0)),
        "npv": float(npv),
        "f1": float(f1_score(y_true, y_prediction, zero_division=0)),
        "brier": float(brier_score_loss(y_true, y_probability)),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

In [ ]:
# Select operating thresholds on validation data while targeting at least 85 percent sensitivity.
validation_lr_probability = logistic_model.predict_proba(
    validation_df[feature_columns]
)[:, 1]

validation_xgb_probability = xgboost_model.predict_proba(
    validation_df[feature_columns]
)[:, 1]

logistic_threshold = choose_threshold_for_sensitivity(
    validation_df["target"],
    validation_lr_probability,
    target_sensitivity=0.85,
)

xgboost_threshold = choose_threshold_for_sensitivity(
    validation_df["target"],
    validation_xgb_probability,
    target_sensitivity=0.85,
)

print("Logistic threshold:", round(logistic_threshold, 4))
print("XGBoost threshold:", round(xgboost_threshold, 4))

In [ ]:
# Evaluate both frozen models on the later temporal test cohort.
test_lr_probability = logistic_model.predict_proba(
    test_df[feature_columns]
)[:, 1]

test_xgb_probability = xgboost_model.predict_proba(
    test_df[feature_columns]
)[:, 1]

logistic_metrics = classification_metrics(
    test_df["target"],
    test_lr_probability,
    logistic_threshold,
)

xgboost_metrics = classification_metrics(
    test_df["target"],
    test_xgb_probability,
    xgboost_threshold,
)

pd.DataFrame(
    [logistic_metrics, xgboost_metrics],
    index=["Logistic Regression", "XGBoost"],
)

In [ ]:
# Plot XGBoost calibration to compare predicted AKI risk with the observed event rate.
observed_rate, predicted_rate = calibration_curve(
    test_df["target"],
    test_xgb_probability,
    n_bins=10,
    strategy="quantile",
)

calibration_table = pd.DataFrame(
    {
        "mean_predicted_probability": predicted_rate,
        "observed_event_rate": observed_rate,
    }
)

plt.figure(figsize=(6, 6))
plt.plot(
    calibration_table["mean_predicted_probability"],
    calibration_table["observed_event_rate"],
    marker="o",
    label="XGBoost",
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed AKI rate")
plt.title("AKI Sentinel calibration")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# Calculate XGBoost performance separately across gender, age, and ICU-type subgroups.
evaluation_df = test_df.copy()
evaluation_df["xgboost_probability"] = test_xgb_probability
evaluation_df["age_group"] = pd.cut(
    evaluation_df["age"],
    bins=[17, 64, 79, float("inf")],
    labels=["18-64", "65-79", "80+"],
)

def subgroup_metrics(dataframe, group_column):
    rows = []

    for group_value, group_df in dataframe.groupby(group_column, dropna=False):
        if len(group_df) < 30 or group_df["target"].nunique() < 2:
            continue

        metrics = classification_metrics(
            group_df["target"],
            group_df["xgboost_probability"],
            xgboost_threshold,
        )

        metrics[group_column] = str(group_value)
        rows.append(metrics)

    return pd.DataFrame(rows)

gender_metrics = subgroup_metrics(evaluation_df, "gender")
age_metrics = subgroup_metrics(evaluation_df, "age_group")
icu_metrics = subgroup_metrics(evaluation_df, "first_careunit")

display(gender_metrics)
display(age_metrics)
display(icu_metrics)

In [ ]:
# Save aggregate evaluation metrics, thresholds, calibration, subgroup results, and private test predictions.
EVALUATION_DIR = DERIVED_DIR / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

with open(EVALUATION_DIR / "test_metrics_logistic_regression.json", "w") as file:
    json.dump(logistic_metrics, file, indent=2)

with open(EVALUATION_DIR / "test_metrics_xgboost.json", "w") as file:
    json.dump(xgboost_metrics, file, indent=2)

with open(EVALUATION_DIR / "thresholds.json", "w") as file:
    json.dump(
        {
            "logistic_regression": logistic_threshold,
            "xgboost": xgboost_threshold,
        },
        file,
        indent=2,
    )

calibration_table.to_csv(
    EVALUATION_DIR / "xgboost_calibration.csv",
    index=False,
)
gender_metrics.to_csv(
    EVALUATION_DIR / "subgroup_gender.csv",
    index=False,
)
age_metrics.to_csv(
    EVALUATION_DIR / "subgroup_age.csv",
    index=False,
)
icu_metrics.to_csv(
    EVALUATION_DIR / "subgroup_icu.csv",
    index=False,
)

private_predictions = test_df[
    ["subject_id", "hadm_id", "stay_id", "target", "anchor_year_group"]
].copy()
private_predictions["xgboost_probability"] = test_xgb_probability
private_predictions.to_parquet(
    EVALUATION_DIR / "private_test_predictions.parquet",
    index=False,
)

print("Saved evaluation outputs.")